# TD3 Self-Driving Car - Google Colab Research Workflow

This notebook is a beginner-friendly, end-to-end Colab workflow for the TD3 car project.

It is compatible with both:
- Google Colab (recommended)
- Local notebook execution (for debugging)

### Workflow Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | **Setup** | Clone repo & install dependencies |
| 2 | **Environment Check** | Verify headless mode, GPU, assets |
| 3 | **Batch 1** | Run experiments 1–3 (basic with N1, N2, N3) |
| 4 | **Batch 2** | Run experiments 4–6 (shaped with N1, N2, N3) |
| 5 | **Batch 3** | Run experiments 7–9 (modified with N1, N2, N3) |
| 6 | **Batch 4** | Run experiments 10–12 (tuned with N1, N2, N3) |
| 7 | **Plot Generation** | Generate all plots from logs |
| 8 | **Save Results** | Organize output into `results/` |
| 9 | **Download Results** | Zip and download everything |

**Total experiments: 12 (4 reward modes × 3 noise levels)**  
**Episodes per experiment: 120**  
**Total episode workload: 1,440 episodes**

> **Tip:** Run batches one at a time to avoid Colab timeouts. Each batch is independent — you can reconnect and continue from where you left off.

---
## 1. Setup — Clone Repo & Install Dependencies

This cell supports both Colab and local execution:
- If the project already exists, it uses it.
- Otherwise, it clones from GitHub.
- Then it switches into the project directory and validates key files.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/adityagangwani30/TD3-Car-Game.git"
DEFAULT_REPO_NAME = "TD3-Car-Game"

def resolve_project_dir() -> Path:
    cwd = Path.cwd()

    # Case 1: already inside the repo
    if (cwd / "main.py").exists() and (cwd / "requirements.txt").exists():
        return cwd

    # Case 2: local workspace has repo folder
    local_candidate = cwd / DEFAULT_REPO_NAME
    if (local_candidate / "main.py").exists():
        return local_candidate

    # Case 3: Colab default path
    colab_base = Path("/content")
    colab_candidate = colab_base / DEFAULT_REPO_NAME
    if colab_candidate.exists():
        return colab_candidate

    # Clone into /content on Colab, otherwise into current working directory
    clone_parent = colab_base if colab_base.exists() else cwd
    target = clone_parent / DEFAULT_REPO_NAME
    if not target.exists():
        print(f"Cloning repository to: {target}")
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    return target

PROJECT_DIR = resolve_project_dir()
os.chdir(PROJECT_DIR)

required_files = ["main.py", "requirements.txt", "run_experiments.py", "environment.py"]
missing = [f for f in required_files if not (PROJECT_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing required files in project directory: {missing}")

print("Current directory:", Path.cwd())
print("Detected project directory:", PROJECT_DIR)
print("Top-level files:", sorted([p.name for p in PROJECT_DIR.iterdir()])[:20])

In [ ]:
import subprocess
import sys

print("Installing/upgrading pip...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Installing project dependencies from requirements.txt...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Import verification
import numpy
import pygame
import torch

print("Dependency check passed")
print("numpy:", numpy.__version__)
print("pygame:", pygame.__version__)
print("torch:", torch.__version__)

---
## 2. Environment Check — Headless Mode & GPU

Colab is headless, so we force SDL to dummy drivers before running simulation code.

This cell also validates:
- GPU availability
- key project imports
- asset generation readiness

In [ ]:
import os
from pathlib import Path

# Force safe headless behavior in Colab/notebook environments
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import torch
from utils import detect_headless_environment, ensure_assets_exist

device = "cuda" if torch.cuda.is_available() else "cpu"
headless_detected = detect_headless_environment()

ensure_assets_exist()

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Using device:", device)
print("Headless detected:", headless_detected)
print("Working directory:", Path.cwd())
print("Assets directory exists:", Path("assets").exists())

# Create output directories upfront
Path("logs").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)
print("Output directories ready (logs/, models/)")

### Quick Sanity Check — Headless Demo

Runs a very short demo to verify the environment works end-to-end. Displays a preview frame if generated.

In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

preview_path = Path("logs/demo_preview.png")
preview_path.parent.mkdir(parents=True, exist_ok=True)

print("Running demo in headless mode...")
subprocess.run([
    sys.executable,
    "main.py",
    "--mode", "demo",
    "--demo-episodes", "2",
    "--headless",
], check=True)

if not preview_path.exists():
    print("Demo preview not found. Generating fallback preview frame...")
    import pygame
    from environment import CarRacingEnv
    from utils import init_pygame

    init_pygame(headless=True)
    env = CarRacingEnv(headless=True, enable_metrics=False)
    env.reset()
    env.render()
    pygame.image.save(env.screen, str(preview_path))
    env.close()
    pygame.quit()

if preview_path.exists():
    print(f"Preview available: {preview_path}")
    display(Image(filename=str(preview_path)))
else:
    raise FileNotFoundError("Preview image was not generated.")

---
## 3–5. Experiment Batches

The full experiment grid has **9 experiments** (3 reward modes × 3 noise levels).

They are split into 3 batches of 3 to **avoid Colab timeouts**:

| Batch | Experiments | Index Range |
|-------|------------|-------------|
| Batch 1 | Experiments 1–3 | `--start-index 0` |
| Batch 2 | Experiments 4–6 | `--start-index 3` |
| Batch 3 | Experiments 7–9 | `--start-index 6` |

> **Important:** Run each batch cell one at a time. If you get disconnected, just reconnect and skip to the next unfinished batch.

### Optional: Live Log Monitor (Run in Parallel)

Use this while batches are running to confirm logs are being written in real time.
Stop this cell anytime with the interrupt button.

In [ ]:
import os
import time

print("Starting log monitor...")
print("Press the stop button to end monitoring.")

try:
    while True:
        print("\n--- Checking logs ---")
        os.system("ls -lah logs")
        time.sleep(30)
except KeyboardInterrupt:
    print("Log monitor stopped by user.")

### Batch 1 — Basic Reward Mode (Experiments 1–3, N1-N3)

In [ ]:
import subprocess
import sys

print("===== BATCH 1: Basic Reward (N1, N2, N3) =====")
print("Experiments 1-3: basic_noise_0.00, basic_noise_0.02, basic_noise_0.05")
print("Running 3 experiments with 120 episodes per experiment")
print(f"Max episodes set to: 120")
print("Command: python -u run_experiments.py --max-experiments 3 --start-index 0 --max-episodes 120 --headless")
print()

process = subprocess.Popen(
    [
        sys.executable, "-u", "run_experiments.py",
        "--max-experiments", "3",
        "--start-index", "0",
        "--max-episodes", "120",
        "--headless",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nSaving logs...")

if return_code == 0:
    print("Batch 1 completed successfully")
else:
    print(f"Batch 1 failed with return code {return_code}")
    print("Error detected. Check the streamed logs above and re-run this batch.")
    raise RuntimeError(f"Batch 1 failed (return code: {return_code})")


### Batch 2 — Shaped Reward Mode (Experiments 4–6, N1-N3)

In [ ]:
import subprocess
import sys

print("===== BATCH 2: Shaped Reward (N1, N2, N3) =====")
print("Experiments 4-6: shaped_noise_0.00, shaped_noise_0.02, shaped_noise_0.05")
print("Running 3 experiments with 120 episodes per experiment")
print(f"Max episodes set to: 120")
print("Command: python -u run_experiments.py --max-experiments 3 --start-index 3 --max-episodes 120 --headless")
print()

process = subprocess.Popen(
    [
        sys.executable, "-u", "run_experiments.py",
        "--max-experiments", "3",
        "--start-index", "3",
        "--max-episodes", "120",
        "--headless",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nSaving logs...")

if return_code == 0:
    print("Batch 2 completed successfully")
else:
    print(f"Batch 2 failed with return code {return_code}")
    print("Error detected. Check the streamed logs above and re-run this batch.")
    raise RuntimeError(f"Batch 2 failed (return code: {return_code})")


### Batch 3 — Modified Reward Mode (Experiments 7–9, N1-N3)

In [ ]:
import subprocess
import sys

print("===== BATCH 3: Modified Reward (N1, N2, N3) =====")
print("Experiments 7-9: modified_noise_0.00, modified_noise_0.02, modified_noise_0.05")
print("Running 3 experiments with 120 episodes per experiment")
print(f"Max episodes set to: 120")
print("Command: python -u run_experiments.py --max-experiments 3 --start-index 6 --max-episodes 120 --headless")
print()

process = subprocess.Popen(
    [
        sys.executable, "-u", "run_experiments.py",
        "--max-experiments", "3",
        "--start-index", "6",
        "--max-episodes", "120",
        "--headless",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nSaving logs...")

if return_code == 0:
    print("Batch 3 completed successfully")
else:
    print(f"Batch 3 failed with return code {return_code}")
    print("Error detected. Check the streamed logs above and re-run this batch.")
    raise RuntimeError(f"Batch 3 failed (return code: {return_code})")


### Verify Experiment Outputs

Quick check that all expected experiment directories and log files were created.

In [ ]:
from pathlib import Path

logs_dir = Path("logs")
models_dir = Path("models")

print("Experiment output verification")
print("=" * 40)

if logs_dir.exists():
    log_subdirs = sorted([p.name for p in logs_dir.iterdir() if p.is_dir()])
    print(f"Log directories ({len(log_subdirs)}): {log_subdirs}")
    
    # Check for training_log.jsonl in each
    for d in log_subdirs:
        log_file = logs_dir / d / "training_log.jsonl"
        status = "OK" if log_file.exists() else "MISSING"
        print(f"  {d}/training_log.jsonl ... {status}")
else:
    print("WARNING: logs/ directory not found!")

print()

if models_dir.exists():
    model_subdirs = sorted([p.name for p in models_dir.iterdir() if p.is_dir()])
    print(f"Model directories ({len(model_subdirs)}): {model_subdirs}")
else:
    print("WARNING: models/ directory not found!")

In [ ]:
import subprocess
import sys

print("===== BATCH 4: Tuned Reward (N1, N2, N3) =====")
print("Experiments 10-12: tuned_noise_0.00, tuned_noise_0.02, tuned_noise_0.05")
print("Running 3 experiments with 120 episodes per experiment")
print(f"Max episodes set to: 120")
print("Command: python -u run_experiments.py --max-experiments 3 --start-index 9 --max-episodes 120 --headless")
print()

process = subprocess.Popen(
    [
        sys.executable, "-u", "run_experiments.py",
        "--max-experiments", "3",
        "--start-index", "9",
        "--max-episodes", "120",
        "--headless",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nSaving logs...")

if return_code == 0:
    print("Batch 4 completed successfully")
else:
    print(f"Batch 4 failed with return code {return_code}")
    print("Error detected. Check the streamed logs above and re-run this batch.")
    raise RuntimeError(f"Batch 4 failed (return code: {return_code})")


### Batch 4 — Tuned Reward Mode (Experiments 10–12, N1-N3)

---
## 6. Plot Generation

Generate publication-ready plots from all experiment logs.

This produces:
- **Per-experiment plots**: reward vs episodes, crash rate, laps completed
- **Comparison plots**: all experiments overlaid for direct comparison

In [ ]:
import subprocess
import sys

print("Generating comparison plots for all experiments...")
print("=" * 50)

result = subprocess.run([
    sys.executable, "plot_metrics.py",
    "--log-dir", "logs",
    "--individual-output", "results/plots/individual",
    "--comparison-output", "results/plots/comparison",
    "--compare",
])

if result.returncode == 0:
    print()
    print("All plots generated successfully!")
else:
    print(f"\nPlot generation returned code {result.returncode}")
    print("Some plots may be missing. Check logs above.")

### Optional: Plot a Specific Experiment

Change the experiment name below to plot any individual experiment:

Available experiment names follow the pattern: `basic_noise_0.00`, `shaped_noise_0.02`, `modified_noise_0.05`, etc.

In [ ]:
import subprocess
import sys

# Change this to plot a specific experiment
EXPERIMENT_TAG = "R1_N1"  # e.g., R1_N1, R2_N2, R3_N3, etc.

print(f"Generating plots for experiment: {EXPERIMENT_TAG}")

subprocess.run([
    sys.executable, "plot_metrics.py",
    "--log-dir", "logs",
    "--experiments", EXPERIMENT_TAG,
    "--individual-output", "results/plots/individual",
    "--comparison-output", "results/plots/comparison",
])

### Display Generated Plots Inline

Show the comparison plots directly in the notebook.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

comparison_dir = Path("results/plots/comparison")

if comparison_dir.exists():
    png_files = sorted(comparison_dir.glob("*.png"))
    if png_files:
        print(f"Found {len(png_files)} comparison plot(s):")
        for png in png_files:
            print(f"  - {png.name}")
            display(Image(filename=str(png), width=800))
    else:
        print("No PNG files found in results/plots/comparison/")
else:
    print("Comparison directory not found. Run the plot generation cell first.")

# Also show any per-experiment plots
individual_dir = Path("results/plots/individual")
exp_pngs = sorted(individual_dir.rglob("*.png")) if individual_dir.exists() else []

if exp_pngs:
    print(f"\nFound {len(exp_pngs)} per-experiment plot(s).")
    # Show first few to avoid overwhelming output
    for png in exp_pngs[:6]:
        print(f"  - {png.relative_to(individual_dir)}")
        display(Image(filename=str(png), width=700))
else:
    print("No per-experiment PNG files found in results/plots/individual/.")

---
## 7. Save Results — Organize Outputs

Build a complete `results/` bundle with:

- `results/plots/comparison/`
- `results/plots/individual/`
- `results/logs/`
- `results/models/`

This cell also migrates any legacy plot locations so no plot PNG remains in the project root.

In [ ]:
import shutil
from pathlib import Path

print("Organizing results...")
print("=" * 40)

results_dir = Path("results")
comparison_dir = results_dir / "plots" / "comparison"
individual_dir = results_dir / "plots" / "individual"
comparison_dir.mkdir(parents=True, exist_ok=True)
individual_dir.mkdir(parents=True, exist_ok=True)

# Migrate any legacy comparison plots from logs/comparison into results/plots/comparison.
legacy_comparison = Path("logs") / "comparison"
if legacy_comparison.exists():
    moved = 0
    for png in legacy_comparison.glob("*.png"):
        target = comparison_dir / png.name
        shutil.move(str(png), str(target))
        moved += 1
    if moved:
        print(f"Moved {moved} legacy comparison plot(s) -> {comparison_dir}")

# Migrate any legacy per-experiment plots from logs/<EXP>/ into results/plots/individual/<EXP>/.
legacy_logs = Path("logs")
if legacy_logs.exists():
    moved = 0
    for exp_dir in sorted(legacy_logs.iterdir()):
        if not exp_dir.is_dir() or exp_dir.name == "comparison":
            continue
        png_files = sorted(exp_dir.glob("*.png"))
        if not png_files:
            continue
        target_exp_dir = individual_dir / exp_dir.name
        target_exp_dir.mkdir(parents=True, exist_ok=True)
        for png in png_files:
            shutil.move(str(png), str(target_exp_dir / png.name))
            moved += 1
    if moved:
        print(f"Moved {moved} legacy per-experiment plot(s) -> {individual_dir}")

# Move any accidental top-level PNGs into results/plots/individual/_misc.
root_pngs = [p for p in Path(".").glob("*.png") if p.is_file()]
if root_pngs:
    misc_dir = individual_dir / "_misc"
    misc_dir.mkdir(parents=True, exist_ok=True)
    for png in root_pngs:
        shutil.move(str(png), str(misc_dir / png.name))
    print(f"Moved {len(root_pngs)} top-level PNG(s) -> {misc_dir}")

# Copy full logs/ and models/ folders into results/.
logs_src = Path("logs")
models_src = Path("models")
logs_dst = results_dir / "logs"
models_dst = results_dir / "models"

if logs_src.exists():
    if logs_dst.exists():
        shutil.rmtree(logs_dst)
    shutil.copytree(logs_src, logs_dst)
    print(f"Copied logs -> {logs_dst}")
else:
    print("WARNING: logs/ not found")

if models_src.exists():
    if models_dst.exists():
        shutil.rmtree(models_dst)
    shutil.copytree(models_src, models_dst)
    print(f"Copied models -> {models_dst}")
else:
    print("WARNING: models/ not found")

comparison_count = sum(1 for _ in comparison_dir.glob("*.png"))
individual_count = sum(1 for _ in individual_dir.rglob("*.png"))
logs_files = sum(1 for _ in logs_dst.rglob("*") if _.is_file()) if logs_dst.exists() else 0
models_files = sum(1 for _ in models_dst.rglob("*") if _.is_file()) if models_dst.exists() else 0

print(f"\nComparison plots: {comparison_count}")
print(f"Individual plots: {individual_count}")
print(f"Logs files: {logs_files}")
print(f"Models files: {models_files}")
print(f"Results ready under: {results_dir}")

---
## 8. Download Results

Create `td3_results.zip` from `results/`.

Archive includes:
- `results/plots/`
- `results/logs/`
- `results/models/`

> **Note:** The `files.download()` call only works in Google Colab. In a local environment, the ZIP file will be created but not auto-downloaded.

In [ ]:
import shutil
from pathlib import Path

results_dir = Path("results")
zip_name = "td3_results"

if not results_dir.exists():
    print("ERROR: results/ is missing. Run the 'Save Results' cell first.")
else:
    has_content = any((results_dir / p).exists() for p in ("plots", "logs", "models"))
    if not has_content:
        print("ERROR: results/ has no plots, logs, or models. Run the previous cells first.")
    else:
        print("Creating ZIP archive...")
        archive_path = shutil.make_archive(zip_name, "zip", str(results_dir))
        zip_size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
        print(f"Archive created: {archive_path} ({zip_size_mb:.1f} MB)")

        # Auto-download on Colab
        try:
            from google.colab import files
            print("Downloading via Colab...")
            files.download(archive_path)
        except ImportError:
            print("Not running in Colab — ZIP file saved locally.")
            print(f"Find it at: {Path(archive_path).resolve()}")